# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR² dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) Python library. All entities—record sets, fields, and columns—are referenced by their unique `@id` fields as recommended by the Croissant standard and this notebook's guidelines.

### Dataset Source
The dataset Croissant schema (JSON-LD) is available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed and up to date
!pip install --quiet --upgrade mlcroissant

## 1. Data Loading

Load the Croissant dataset metadata and access the main description and high-level fields. All metadata entities can be referenced by their `@id` fields for detailed programmatic access.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic info
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview

Let's inspect the available record sets in the dataset using their `@id` fields. We will then review the fields within each record set by their `@id` as well.

To ensure we reference entities by their `@id`, we will extract the relevant IDs for record sets and their fields.

In [ ]:
# List all record sets with '@id' and fields with '@id'.
print('RecordSets in dataset:')
record_sets = []
for rset in dataset.record_sets:
    print(f"- @id: {rset.id}, name: {rset.name}")
    record_sets.append(rset.id)

    print('  Fields:')
    for field in rset.fields:
        print(f"    - field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")
print('\nAll RecordSet @id values:', record_sets)

## 3. Data Extraction

We will extract tables from each record set using their `@id` fields with `mlcroissant`, and load data into Pandas DataFrames for further analysis.

**Example below uses the first discovered record set.**

Make sure to adjust which record set or field IDs you use for your purposes.

In [ ]:
# Load all record sets into DataFrames using @id
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}, shape: {df.shape}")

# Let's explore the first record set in more detail
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id is not None:
    print(f"\nColumns in {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.to_list())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's process and analyze the main table:
- We'll select a numeric field (e.g. protein abundance or peptide count) using its field `@id`
- Demonstrate filtering, normalization, and grouping by another column
- All references will use column/field `@id` for reproducibility and standardization

Update the IDs to reflect those discovered in the dataset for your analysis.

In [ ]:
# Use the main record set for EDA
df = dataframes[main_record_set_id]

# Choose a numeric field @id. List all columns to help select:
print("Columns available (likely @id):", df.columns.tolist())

# Example: Use first numeric field (fallback to a plausible field such as 'cr:PeptideCount')
candidate_numeric_ids = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[col])]
if candidate_numeric_ids:
    numeric_field_id = candidate_numeric_ids[0]
else:
    # Fallback example
    numeric_field_id = 'cr:PeptideCount'

print(f"Selected numeric field: {numeric_field_id}")

# Filter for values above a threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Try grouping by a categorical field (e.g. protein type), using @id.
possible_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
group_field_id = possible_group_fields[0] if possible_group_fields else None
if group_field_id:
    print(f"\nGrouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and the normalized values, and (if applicable) the grouping result. Visualizations use fields referenced by their `@id` for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.xlabel(numeric_field_id)
plt.show()

# Show normalized
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id + '_normalized'], bins=30, kde=True)
plt.title(f"Normalized {numeric_field_id} Distribution (> {threshold})")
plt.xlabel(numeric_field_id + '_normalized')
plt.show()

# Visualize group means if available
if group_field_id:
    plt.figure(figsize=(10, 5))
    grouped_df.plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we:
- Explored the FAIR² dataset using the `mlcroissant` library
- Inspected record sets, fields, and columns by their `@id` references
- Loaded tabular data into Pandas
- Filtered, normalized, and visualized key numeric fields for further downstream analysis

Adjust the field and record set IDs as needed based on your use case and the entities in your specific dataset instance!
